# Stress Prediction v7 — Compact 16 Features

1등 팀 전략을 참고한 **압축형 피처 실험 버전**입니다.

핵심:

- 총 피처 수를 정확히 16개로 제한
- 원본 9개 + 파생 7개 구조
- `mean_working` 직접 입력 제외
- BMI, 맥압, 평균동맥압, 혈압비, 결측 플래그, 범주 조합 2개 사용
- 피처 복제 유지/제거 비교
- 혈압 원본 유지/제거 비교
- 기존 ExtraTrees + Pair + Override 구조 유지
- 가장 좋은 compact feature set만 최종 제출에 사용


In [1]:
from __future__ import annotations

import os
import warnings
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_SPLITS = 5

SCREEN_N_ESTIMATORS = 700
FINAL_N_ESTIMATORS = 1200

TREE_QUANTILES = [0.50, 0.52, 0.54]
PAIR_QUANTILES = [0.46, 0.48, 0.50]
PAIR_WEIGHTS = [0.15, 0.20, 0.25]
OVERRIDE_THRESHOLDS = [0.15, 0.18, 0.20, None]
ROUND_OPTIONS = [2, None]

WINSORIZE_COLUMNS = [
    "mean_working",
    "diastolic_blood_pressure",
    "cholesterol",
    "glucose",
]

# 기존 최고 구조의 복제 설정
FEATURE_COPY_COUNTS = {
    "mean_working": 1,
    "bmi": 4,
    "cholesterol": 2,
    "height": 2,
    "glucose": 3,
    "weight": 2,
    "cholesterol_glucose_ratio": 2,
    "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working",
    "bmi",
    "cholesterol",
    "height",
    "glucose",
    "weight",
    "cholesterol_glucose_ratio",
    "bone_density",
]

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project/8_6")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FALLBACK = Path("/mnt/data/train(2).csv")
TEST_FALLBACK = Path("/mnt/data/test(2).csv")

print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


DATA_DIR: /Users/bitsaem/Desktop/stress_project/open
OUTPUT_DIR: /Users/bitsaem/Desktop/stress_project/8_6


## 1. 데이터 로드


In [2]:
train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"
sample_path = DATA_DIR / "sample_submission.csv"

if not train_path.exists() and TRAIN_FALLBACK.exists():
    train_path = TRAIN_FALLBACK

if not test_path.exists() and TEST_FALLBACK.exists():
    test_path = TEST_FALLBACK

train_raw = pd.read_csv(train_path)
test_raw = pd.read_csv(test_path)

y = train_raw[TARGET].reset_index(drop=True)
X_raw = (
    train_raw
    .drop(columns=[TARGET, ID_COLUMN])
    .reset_index(drop=True)
)
X_test_raw = (
    test_raw
    .drop(columns=[ID_COLUMN])
    .reset_index(drop=True)
)

numeric_raw_columns = X_raw.select_dtypes(include=["number"]).columns.tolist()

print("Train:", train_raw.shape)
print("Test :", test_raw.shape)
display(train_raw.head())


Train: (3000, 18)
Test : (3000, 17)


,ID,gender,age,height,weight,cholesterol,systolic_blood_pressure,diastolic_blood_pressure,glucose,bone_density,activity,smoke_status,medical_history,family_medical_history,sleep_pattern,edu_level,mean_working,stress_score
0,TRAIN_0000,F,72,161.49,58.47,279.84,165,100,143.35,0.87,moderate,ex-smoker,high blood pressure,diabetes,sleep difficulty,bachelors degree,NaN,0.63
1,TRAIN_0001,M,88,179.87,77.60,257.37,178,111,146.94,0.07,moderate,ex-smoker,NaN,diabetes,normal,graduate degree,NaN,0.83
2,TRAIN_0002,M,47,182.47,89.93,226.66,134,95,142.61,1.18,light,ex-smoker,NaN,NaN,normal,high school diploma,9.0,0.70
3,TRAIN_0003,M,69,185.78,68.63,206.74,158,92,137.26,0.48,intense,ex-smoker,high blood pressure,NaN,oversleeping,graduate degree,NaN,0.17
4,TRAIN_0004,F,81,164.63,71.53,255.92,171,116,129.37,0.34,moderate,ex-smoker,diabetes,diabetes,sleep difficulty,bachelors degree,NaN,0.36


## 2. 파생 피처와 16개 compact 후보


In [3]:
def safe_divide(a: pd.Series, b: pd.Series) -> pd.Series:
    return a / (b.abs() + 1e-6)


def add_compact_derived(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()

    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]

    result["bmi"] = safe_divide(
        result["weight"],
        height_m.pow(2),
    )
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = safe_divide(sbp, dbp)
    result["missing_flag"] = frame.isna().any(axis=1).astype(int)

    result["activity_sleep_cross"] = (
        result["activity"].fillna("__MISSING__").astype(str)
        + "__"
        + result["sleep_pattern"].fillna("__MISSING__").astype(str)
    )

    result["medical_family_cross"] = (
        result["medical_history"].fillna("__MISSING__").astype(str)
        + "__"
        + result["family_medical_history"].fillna("__MISSING__").astype(str)
    )

    return result


DERIVED_7 = [
    "bmi",
    "pulse_pressure",
    "mean_arterial_pressure",
    "blood_pressure_ratio",
    "missing_flag",
    "activity_sleep_cross",
    "medical_family_cross",
]

# 원본 9개 후보 조합
ORIGINAL_9_SETS = {
    # 수치형 중심
    "numeric9": [
        "age",
        "height",
        "weight",
        "cholesterol",
        "systolic_blood_pressure",
        "diastolic_blood_pressure",
        "glucose",
        "bone_density",
        "activity",
    ],

    # 행동/수면 포함
    "behavior9": [
        "age",
        "height",
        "weight",
        "cholesterol",
        "systolic_blood_pressure",
        "diastolic_blood_pressure",
        "glucose",
        "activity",
        "sleep_pattern",
    ],

    # 병력 포함
    "medical9": [
        "age",
        "height",
        "weight",
        "cholesterol",
        "systolic_blood_pressure",
        "diastolic_blood_pressure",
        "glucose",
        "medical_history",
        "family_medical_history",
    ],

    # bone + 행동/수면
    "balanced9": [
        "age",
        "height",
        "weight",
        "cholesterol",
        "glucose",
        "bone_density",
        "activity",
        "sleep_pattern",
        "medical_history",
    ],
}


def build_compact_features(
    frame: pd.DataFrame,
    original_set_name: str,
    drop_bp_raw: bool = False,
    include_mean_working: bool = False,
) -> pd.DataFrame:
    enriched = add_compact_derived(frame)
    original_cols = ORIGINAL_9_SETS[original_set_name].copy()

    if drop_bp_raw:
        original_cols = [
            col for col in original_cols
            if col not in {
                "systolic_blood_pressure",
                "diastolic_blood_pressure",
            }
        ]

        # 정확히 9개 유지하기 위해 다른 원본 추가
        replacements = [
            "activity",
            "sleep_pattern",
            "medical_history",
            "family_medical_history",
            "edu_level",
            "smoke_status",
            "bone_density",
        ]
        for col in replacements:
            if col not in original_cols:
                original_cols.append(col)
            if len(original_cols) == 9:
                break

    if include_mean_working:
        # 9개를 유지하기 위해 마지막 컬럼 대신 mean_working
        if "mean_working" not in original_cols:
            original_cols = original_cols[:8] + ["mean_working"]

    selected = original_cols + DERIVED_7
    result = enriched[selected].copy()

    if result.shape[1] != 16:
        raise ValueError(
            f"{original_set_name}: 피처 수가 16개가 아닙니다: {result.shape[1]}"
        )

    return result


COMPACT_CONFIGS = []

for original_set_name in ORIGINAL_9_SETS:
    COMPACT_CONFIGS.extend([
        {
            "name": f"{original_set_name}_copy_on",
            "original_set_name": original_set_name,
            "drop_bp_raw": False,
            "include_mean_working": False,
            "use_feature_copy": True,
        },
        {
            "name": f"{original_set_name}_copy_off",
            "original_set_name": original_set_name,
            "drop_bp_raw": False,
            "include_mean_working": False,
            "use_feature_copy": False,
        },
        {
            "name": f"{original_set_name}_drop_bp",
            "original_set_name": original_set_name,
            "drop_bp_raw": True,
            "include_mean_working": False,
            "use_feature_copy": False,
        },
        {
            "name": f"{original_set_name}_with_working",
            "original_set_name": original_set_name,
            "drop_bp_raw": False,
            "include_mean_working": True,
            "use_feature_copy": False,
        },
    ])

print("Compact 후보 수:", len(COMPACT_CONFIGS))

for config in COMPACT_CONFIGS[:4]:
    sample = build_compact_features(
        X_raw.head(5),
        config["original_set_name"],
        config["drop_bp_raw"],
        config["include_mean_working"],
    )
    print(config["name"], sample.shape, sample.columns.tolist())


Compact 후보 수: 16
numeric9_copy_on (5, 16) ['age', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure', 'diastolic_blood_pressure', 'glucose', 'bone_density', 'activity', 'bmi', 'pulse_pressure', 'mean_arterial_pressure', 'blood_pressure_ratio', 'missing_flag', 'activity_sleep_cross', 'medical_family_cross']
numeric9_copy_off (5, 16) ['age', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure', 'diastolic_blood_pressure', 'glucose', 'bone_density', 'activity', 'bmi', 'pulse_pressure', 'mean_arterial_pressure', 'blood_pressure_ratio', 'missing_flag', 'activity_sleep_cross', 'medical_family_cross']
numeric9_drop_bp (5, 16) ['age', 'height', 'weight', 'cholesterol', 'glucose', 'bone_density', 'activity', 'sleep_pattern', 'medical_history', 'bmi', 'pulse_pressure', 'mean_arterial_pressure', 'blood_pressure_ratio', 'missing_flag', 'activity_sleep_cross', 'medical_family_cross']
numeric9_with_working (5, 16) ['age', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure'

## 3. ExtraTrees / Pair / Override 함수


In [4]:
def compute_iqr_bounds(train_frame, columns):
    bounds = {}
    for col in columns:
        q1, q3 = train_frame[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        bounds[col] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)
    return bounds


def apply_winsorize(frame, bounds):
    result = frame.copy()
    for col, (lower, upper) in bounds.items():
        result[col] = result[col].clip(lower, upper)
    return result


def add_feature_copies(frame):
    result = frame.copy()
    for feature, n_copies in FEATURE_COPY_COUNTS.items():
        if feature not in result.columns:
            continue
        for i in range(1, n_copies + 1):
            result[f"{feature}__copy{i}"] = result[feature]
    return result


def make_preprocessor(frame):
    categorical = frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    numerical = frame.select_dtypes(
        include=["number"]
    ).columns.tolist()

    return ColumnTransformer([
        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="constant",
                        fill_value="missing",
                    ),
                ),
                (
                    "encoder",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False,
                    ),
                ),
            ]),
            categorical,
        ),
        (
            "numerical",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
            ),
            numerical,
        ),
    ])


def tree_prediction_matrix(
    train_features,
    val_features,
    target,
    use_feature_copy,
    n_estimators,
):
    train_input = train_features.copy()
    val_input = val_features.copy()

    if use_feature_copy:
        train_input = add_feature_copies(train_input)
        val_input = add_feature_copies(val_input)

    preprocessor = make_preprocessor(train_input)
    train_matrix = preprocessor.fit_transform(train_input)
    val_matrix = preprocessor.transform(val_input)

    model = ExtraTreesRegressor(
        n_estimators=n_estimators,
        min_samples_leaf=1,
        max_features=1,
        criterion="squared_error",
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    model.fit(train_matrix, target)

    return np.column_stack([
        tree.predict(val_matrix)
        for tree in model.estimators_
    ])


def add_base_pair_features(frame):
    result = frame.copy()
    height_m = result["height"] / 100.0
    result["bmi"] = safe_divide(
        result["weight"],
        height_m.pow(2),
    )
    result["cholesterol_glucose_ratio"] = safe_divide(
        result["cholesterol"],
        result["glucose"],
    )
    return result


def empirical_rank_transform(train_values, test_values):
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)

    sorted_train = np.sort(train_values)
    train_rank = (
        np.searchsorted(sorted_train, train_values, side="right")
        / len(train_values)
    )
    test_rank = (
        np.searchsorted(sorted_train, test_values, side="right")
        / len(train_values)
    )

    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def pair_prediction_matrix(train_features, val_features, target):
    train_rank = {}
    val_rank = {}

    for col in PAIR_COLUMNS:
        train_rank[col], val_rank[col] = empirical_rank_transform(
            train_features[col].to_numpy(dtype=float),
            val_features[col].to_numpy(dtype=float),
        )

    target_values = target.to_numpy(dtype=float)
    pair_targets = []

    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(
                val_rank[first][:, None]
                - train_rank[first][None, :]
            )
            + np.abs(
                val_rank[second][:, None]
                - train_rank[second][None, :]
            )
        )
        nearest_idx = np.argmin(distance, axis=1)
        pair_targets.append(target_values[nearest_idx])

    return np.column_stack(pair_targets)


def nearest_neighbor_lookup(train_raw_frame, val_raw_frame, target):
    train_numeric = train_raw_frame[numeric_raw_columns].copy()
    val_numeric = val_raw_frame[numeric_raw_columns].copy()

    medians = train_numeric.median()
    train_numeric = train_numeric.fillna(medians)
    val_numeric = val_numeric.fillna(medians)

    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_numeric)
    val_scaled = scaler.transform(val_numeric)

    tree = cKDTree(train_scaled)
    nearest_dist, nearest_idx = tree.query(val_scaled, k=1)

    nearest_target = target.to_numpy(dtype=float)[nearest_idx]
    return nearest_dist, nearest_target


def apply_postprocess(
    prediction,
    round_decimals,
    override_threshold,
    nearest_dist,
    nearest_target,
):
    result = np.clip(prediction.copy(), 0, 1)

    if round_decimals is not None:
        result = np.round(result, round_decimals)

    if override_threshold is not None:
        mask = nearest_dist < override_threshold
        replacement = nearest_target[mask].copy()

        if round_decimals is not None:
            replacement = np.round(replacement, round_decimals)

        result[mask] = np.clip(replacement, 0, 1)

    return result


## 4. Compact 16개 후보별 OOF 비교


In [5]:
cv_splits = list(
    KFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_SEED,
    ).split(X_raw)
)

screen_rows = []
screen_cache = {}

for config in COMPACT_CONFIGS:
    oof_pred = np.zeros(len(y))

    for fold, (train_idx, val_idx) in enumerate(cv_splits, start=1):
        train_fold_raw = X_raw.iloc[train_idx].reset_index(drop=True)
        val_fold_raw = X_raw.iloc[val_idx].reset_index(drop=True)

        y_train = y.iloc[train_idx].reset_index(drop=True)

        bounds = compute_iqr_bounds(
            train_fold_raw,
            WINSORIZE_COLUMNS,
        )

        train_w = apply_winsorize(train_fold_raw, bounds)
        val_w = apply_winsorize(val_fold_raw, bounds)

        train_compact = build_compact_features(
            train_w,
            config["original_set_name"],
            config["drop_bp_raw"],
            config["include_mean_working"],
        )
        val_compact = build_compact_features(
            val_w,
            config["original_set_name"],
            config["drop_bp_raw"],
            config["include_mean_working"],
        )

        tree_matrix = tree_prediction_matrix(
            train_compact,
            val_compact,
            y_train,
            use_feature_copy=config["use_feature_copy"],
            n_estimators=SCREEN_N_ESTIMATORS,
        )

        pred = np.quantile(tree_matrix, 0.52, axis=1)
        pred = np.round(np.clip(pred, 0, 1), 2)

        oof_pred[val_idx] = pred

        print(
            f"[{config['name']}] Fold {fold}/{N_SPLITS} 완료"
        )

    mae = mean_absolute_error(y, oof_pred)

    screen_rows.append({
        "config_name": config["name"],
        "original_set_name": config["original_set_name"],
        "drop_bp_raw": config["drop_bp_raw"],
        "include_mean_working": config["include_mean_working"],
        "use_feature_copy": config["use_feature_copy"],
        "feature_count": 16,
        "cv_mae": mae,
    })

    screen_cache[config["name"]] = oof_pred
    print(f"{config['name']}: CV MAE={mae:.8f}\n")

screen_df = (
    pd.DataFrame(screen_rows)
    .sort_values("cv_mae")
    .reset_index(drop=True)
)

display(screen_df)


[numeric9_copy_on] Fold 1/5 완료
[numeric9_copy_on] Fold 2/5 완료
[numeric9_copy_on] Fold 3/5 완료
[numeric9_copy_on] Fold 4/5 완료
[numeric9_copy_on] Fold 5/5 완료
numeric9_copy_on: CV MAE=0.15170667

[numeric9_copy_off] Fold 1/5 완료
[numeric9_copy_off] Fold 2/5 완료
[numeric9_copy_off] Fold 3/5 완료
[numeric9_copy_off] Fold 4/5 완료
[numeric9_copy_off] Fold 5/5 완료
numeric9_copy_off: CV MAE=0.15745667

[numeric9_drop_bp] Fold 1/5 완료
[numeric9_drop_bp] Fold 2/5 완료
[numeric9_drop_bp] Fold 3/5 완료
[numeric9_drop_bp] Fold 4/5 완료
[numeric9_drop_bp] Fold 5/5 완료
numeric9_drop_bp: CV MAE=0.16040667

[numeric9_with_working] Fold 1/5 완료
[numeric9_with_working] Fold 2/5 완료
[numeric9_with_working] Fold 3/5 완료
[numeric9_with_working] Fold 4/5 완료
[numeric9_with_working] Fold 5/5 완료
numeric9_with_working: CV MAE=0.15593333

[behavior9_copy_on] Fold 1/5 완료
[behavior9_copy_on] Fold 2/5 완료
[behavior9_copy_on] Fold 3/5 완료
[behavior9_copy_on] Fold 4/5 완료
[behavior9_copy_on] Fold 5/5 완료
behavior9_copy_on: CV MAE=0.15329333

,config_name,original_set_name,drop_bp_raw,include_mean_working,use_feature_copy,feature_count,cv_mae
0,numeric9_copy_on,numeric9,False,False,True,16,0.151707
1,balanced9_copy_on,balanced9,False,False,True,16,0.152983
2,medical9_copy_on,medical9,False,False,True,16,0.153007
3,behavior9_copy_on,behavior9,False,False,True,16,0.153293
4,numeric9_with_working,numeric9,False,True,False,16,0.155933
5,numeric9_copy_off,numeric9,False,False,False,16,0.157457
6,behavior9_with_working,behavior9,False,True,False,16,0.158350
7,balanced9_with_working,balanced9,False,True,False,16,0.158760
8,medical9_with_working,medical9,False,True,False,16,0.159450
9,medical9_copy_off,medical9,False,False,False,16,0.159467


## 5. 최적 compact 설정으로 Pair/Quantile/Override 탐색


In [6]:
best_row = screen_df.iloc[0]

BEST_COMPACT_CONFIG = {
    "name": best_row["config_name"],
    "original_set_name": best_row["original_set_name"],
    "drop_bp_raw": bool(best_row["drop_bp_raw"]),
    "include_mean_working": bool(best_row["include_mean_working"]),
    "use_feature_copy": bool(best_row["use_feature_copy"]),
}

print("선택된 compact 설정:")
print(BEST_COMPACT_CONFIG)

fold_cache = []

for fold, (train_idx, val_idx) in enumerate(cv_splits, start=1):
    train_fold_raw = X_raw.iloc[train_idx].reset_index(drop=True)
    val_fold_raw = X_raw.iloc[val_idx].reset_index(drop=True)

    y_train = y.iloc[train_idx].reset_index(drop=True)

    bounds = compute_iqr_bounds(
        train_fold_raw,
        WINSORIZE_COLUMNS,
    )
    train_w = apply_winsorize(train_fold_raw, bounds)
    val_w = apply_winsorize(val_fold_raw, bounds)

    train_compact = build_compact_features(
        train_w,
        BEST_COMPACT_CONFIG["original_set_name"],
        BEST_COMPACT_CONFIG["drop_bp_raw"],
        BEST_COMPACT_CONFIG["include_mean_working"],
    )
    val_compact = build_compact_features(
        val_w,
        BEST_COMPACT_CONFIG["original_set_name"],
        BEST_COMPACT_CONFIG["drop_bp_raw"],
        BEST_COMPACT_CONFIG["include_mean_working"],
    )

    tree_matrix = tree_prediction_matrix(
        train_compact,
        val_compact,
        y_train,
        use_feature_copy=BEST_COMPACT_CONFIG["use_feature_copy"],
        n_estimators=FINAL_N_ESTIMATORS,
    )

    train_pair = add_base_pair_features(train_w)
    val_pair = add_base_pair_features(val_w)

    pair_matrix = pair_prediction_matrix(
        train_pair,
        val_pair,
        y_train,
    )

    nearest_dist, nearest_target = nearest_neighbor_lookup(
        train_fold_raw,
        val_fold_raw,
        y_train,
    )

    fold_cache.append({
        "val_idx": val_idx,
        "tree_matrix": tree_matrix,
        "pair_matrix": pair_matrix,
        "nearest_dist": nearest_dist,
        "nearest_target": nearest_target,
    })

    print(f"Fold {fold}/{N_SPLITS} 캐시 완료")


선택된 compact 설정:
{'name': 'numeric9_copy_on', 'original_set_name': 'numeric9', 'drop_bp_raw': False, 'include_mean_working': False, 'use_feature_copy': True}
Fold 1/5 캐시 완료
Fold 2/5 캐시 완료
Fold 3/5 캐시 완료
Fold 4/5 캐시 완료
Fold 5/5 캐시 완료


In [7]:
search_rows = []

for tree_q in TREE_QUANTILES:
    for pair_q in PAIR_QUANTILES:
        for pair_weight in PAIR_WEIGHTS:
            for decimals in ROUND_OPTIONS:
                for threshold in OVERRIDE_THRESHOLDS:
                    oof_pred = np.zeros(len(y))

                    for cache in fold_cache:
                        tree_pred = np.quantile(
                            cache["tree_matrix"],
                            tree_q,
                            axis=1,
                        )
                        pair_pred = np.quantile(
                            cache["pair_matrix"],
                            pair_q,
                            axis=1,
                        )

                        blended = (
                            (1 - pair_weight) * tree_pred
                            + pair_weight * pair_pred
                        )

                        final_pred = apply_postprocess(
                            blended,
                            decimals,
                            threshold,
                            cache["nearest_dist"],
                            cache["nearest_target"],
                        )

                        oof_pred[cache["val_idx"]] = final_pred

                    mae = mean_absolute_error(y, oof_pred)

                    search_rows.append({
                        "cv_mae": mae,
                        "tree_q": tree_q,
                        "pair_q": pair_q,
                        "pair_weight": pair_weight,
                        "round_decimals": decimals,
                        "override_threshold": threshold,
                        "compact_config": BEST_COMPACT_CONFIG["name"],
                    })

search_df = (
    pd.DataFrame(search_rows)
    .sort_values("cv_mae")
    .reset_index(drop=True)
)

display(search_df.head(30))
print("최적 OOF MAE:", search_df.iloc[0]["cv_mae"])


,cv_mae,tree_q,pair_q,pair_weight,round_decimals,override_threshold,compact_config
0,0.150403,0.52,0.48,0.25,2.0,0.15,numeric9_copy_on
1,0.150403,0.52,0.48,0.25,2.0,0.18,numeric9_copy_on
2,0.150403,0.52,0.48,0.25,2.0,0.20,numeric9_copy_on
3,0.150403,0.52,0.48,0.25,2.0,NaN,numeric9_copy_on
4,0.150479,0.52,0.48,0.25,NaN,0.15,numeric9_copy_on
5,0.150479,0.52,0.48,0.25,NaN,0.18,numeric9_copy_on
6,0.150479,0.52,0.48,0.25,NaN,0.20,numeric9_copy_on
7,0.150479,0.52,0.48,0.25,NaN,NaN,numeric9_copy_on
8,0.150536,0.52,0.50,0.25,NaN,NaN,numeric9_copy_on
9,0.150536,0.52,0.50,0.25,NaN,0.18,numeric9_copy_on


최적 OOF MAE: 0.15040333333333333


## 6. 제출 후보 생성


In [8]:
def row_to_config(row):
    return {
        "tree_q": float(row["tree_q"]),
        "pair_q": float(row["pair_q"]),
        "pair_weight": float(row["pair_weight"]),
        "round_decimals": (
            None
            if pd.isna(row["round_decimals"])
            else int(row["round_decimals"])
        ),
        "override_threshold": (
            None
            if pd.isna(row["override_threshold"])
            else float(row["override_threshold"])
        ),
        "cv_mae": float(row["cv_mae"]),
    }


candidate_rows = [
    ("cv_best", search_df.iloc[0]),
]

safe = search_df[
    (search_df["tree_q"] == 0.52)
    & (search_df["pair_q"] == 0.48)
    & (search_df["pair_weight"] == 0.20)
    & (search_df["round_decimals"] == 2)
]
if len(safe):
    candidate_rows.append(("safe", safe.iloc[0]))

no_override = search_df[
    search_df["override_threshold"].isna()
]
if len(no_override):
    candidate_rows.append(("no_override", no_override.iloc[0]))

pair_heavy = search_df[
    search_df["pair_weight"] == max(PAIR_WEIGHTS)
]
if len(pair_heavy):
    candidate_rows.append(("pair_heavy", pair_heavy.iloc[0]))

tree_heavy = search_df[
    search_df["pair_weight"] == min(PAIR_WEIGHTS)
]
if len(tree_heavy):
    candidate_rows.append(("tree_heavy", tree_heavy.iloc[0]))

FINAL_CONFIGS = []
seen = set()

for name, row in candidate_rows:
    config = row_to_config(row)
    key = tuple(config.items())

    if key not in seen:
        seen.add(key)
        FINAL_CONFIGS.append((name, config))

FINAL_CONFIGS = FINAL_CONFIGS[:5]

print("최종 제출 후보:")
for name, config in FINAL_CONFIGS:
    print(name, config)


최종 제출 후보:
cv_best {'tree_q': 0.52, 'pair_q': 0.48, 'pair_weight': 0.25, 'round_decimals': 2, 'override_threshold': 0.15, 'cv_mae': 0.15040333333333333}
safe {'tree_q': 0.52, 'pair_q': 0.48, 'pair_weight': 0.2, 'round_decimals': 2, 'override_threshold': None, 'cv_mae': 0.15064}
no_override {'tree_q': 0.52, 'pair_q': 0.48, 'pair_weight': 0.25, 'round_decimals': 2, 'override_threshold': None, 'cv_mae': 0.15040333333333333}
tree_heavy {'tree_q': 0.52, 'pair_q': 0.48, 'pair_weight': 0.15, 'round_decimals': None, 'override_threshold': 0.2, 'cv_mae': 0.15085811333333335}


In [9]:
if not sample_path.exists():
    submission_template = pd.DataFrame({
        ID_COLUMN: test_raw[ID_COLUMN],
        TARGET: 0.0,
    })
else:
    submission_template = pd.read_csv(sample_path)

bounds = compute_iqr_bounds(X_raw, WINSORIZE_COLUMNS)
train_w = apply_winsorize(X_raw, bounds)
test_w = apply_winsorize(X_test_raw, bounds)

train_compact = build_compact_features(
    train_w,
    BEST_COMPACT_CONFIG["original_set_name"],
    BEST_COMPACT_CONFIG["drop_bp_raw"],
    BEST_COMPACT_CONFIG["include_mean_working"],
)
test_compact = build_compact_features(
    test_w,
    BEST_COMPACT_CONFIG["original_set_name"],
    BEST_COMPACT_CONFIG["drop_bp_raw"],
    BEST_COMPACT_CONFIG["include_mean_working"],
)

tree_matrix_test = tree_prediction_matrix(
    train_compact,
    test_compact,
    y,
    use_feature_copy=BEST_COMPACT_CONFIG["use_feature_copy"],
    n_estimators=FINAL_N_ESTIMATORS,
)

train_pair = add_base_pair_features(train_w)
test_pair = add_base_pair_features(test_w)

pair_matrix_test = pair_prediction_matrix(
    train_pair,
    test_pair,
    y,
)

nearest_dist_test, nearest_target_test = nearest_neighbor_lookup(
    X_raw,
    X_test_raw,
    y,
)

saved_paths = []

for rank, (name, config) in enumerate(FINAL_CONFIGS, start=1):
    tree_pred = np.quantile(
        tree_matrix_test,
        config["tree_q"],
        axis=1,
    )
    pair_pred = np.quantile(
        pair_matrix_test,
        config["pair_q"],
        axis=1,
    )

    blended = (
        (1 - config["pair_weight"]) * tree_pred
        + config["pair_weight"] * pair_pred
    )

    final_pred = apply_postprocess(
        blended,
        config["round_decimals"],
        config["override_threshold"],
        nearest_dist_test,
        nearest_target_test,
    )

    submission = submission_template.copy()
    submission[TARGET] = final_pred

    filename = (
        f"submit_compact16_v7_rank{rank}_{name}"
        f"_{BEST_COMPACT_CONFIG['name']}.csv"
    )

    output_file = OUTPUT_DIR / filename
    submission.to_csv(output_file, index=False)

    saved_paths.append(output_file)

    print("\nSaved:", output_file)
    print("Config:", config)
    display(submission.head())

print("\n생성 완료")
for path in saved_paths:
    print(path)



Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_compact16_v7_rank1_cv_best_numeric9_copy_on.csv
Config: {'tree_q': 0.52, 'pair_q': 0.48, 'pair_weight': 0.25, 'round_decimals': 2, 'override_threshold': 0.15, 'cv_mae': 0.15040333333333333}


,ID,stress_score
0,TEST_0000,0.58
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.55
4,TEST_0004,0.53



Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_compact16_v7_rank2_safe_numeric9_copy_on.csv
Config: {'tree_q': 0.52, 'pair_q': 0.48, 'pair_weight': 0.2, 'round_decimals': 2, 'override_threshold': None, 'cv_mae': 0.15064}


,ID,stress_score
0,TEST_0000,0.58
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.55
4,TEST_0004,0.53



Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_compact16_v7_rank3_no_override_numeric9_copy_on.csv
Config: {'tree_q': 0.52, 'pair_q': 0.48, 'pair_weight': 0.25, 'round_decimals': 2, 'override_threshold': None, 'cv_mae': 0.15040333333333333}


,ID,stress_score
0,TEST_0000,0.58
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.55
4,TEST_0004,0.53



Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_compact16_v7_rank4_tree_heavy_numeric9_copy_on.csv
Config: {'tree_q': 0.52, 'pair_q': 0.48, 'pair_weight': 0.15, 'round_decimals': None, 'override_threshold': 0.2, 'cv_mae': 0.15085811333333335}


,ID,stress_score
0,TEST_0000,0.57814
1,TEST_0001,0.97000
2,TEST_0002,0.19000
3,TEST_0003,0.55144
4,TEST_0004,0.53000



생성 완료
/Users/bitsaem/Desktop/stress_project/8_6/submit_compact16_v7_rank1_cv_best_numeric9_copy_on.csv
/Users/bitsaem/Desktop/stress_project/8_6/submit_compact16_v7_rank2_safe_numeric9_copy_on.csv
/Users/bitsaem/Desktop/stress_project/8_6/submit_compact16_v7_rank3_no_override_numeric9_copy_on.csv
/Users/bitsaem/Desktop/stress_project/8_6/submit_compact16_v7_rank4_tree_heavy_numeric9_copy_on.csv
